Group Members: Nicolas Banatt, Annanya Jain, James McDermott, Yanran Jia

In [15]:
!python --version

Python 3.13.7


In [16]:
%pip install pandas


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd

In [18]:
df_ag = pd.read_csv('./AGData.csv') # ActiGraph accelerometer data
df_rrv = pd.read_csv('./RRVData.csv') # Relative reinforcing value data

df_ag.head()

,Participant,ID,AG_Date,Completed,Gender,Group,Weight,Height,Race,Ethnicity,...,Avg_Daily_Light_Sa_Su_Min,Avg_Daily_Mod_Week_Min,Avg_Daily_Mod_M_F_Min,Avg_Daily_Mod_Sa_Su_Min,Avg_Daily_Vig_Week_Min,Avg_Daily_Vig_M_F_Min,Avg_Daily_Vig_Sa_Su_Min,Avg_Daily_Very_Vig_Week_Min,Avg_Daily_Very_Vig_M_F_Min,Avg_Daily_Very_Vig_Sa_Su_Min
0,407-0001,1,11-Dec-17,Y,M,control,NaN,NaN,White,Not Hispanic or Latino,...,264.5,22.0,26.0,13,3.0,4.0,0,0.0,0.0,0
1,407-0001,1,14-Dec-17,Y,M,control,129.4,196.0,White,Not Hispanic or Latino,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,407-0001,1,5-Mar-18,Y,M,control,NaN,NaN,White,Not Hispanic or Latino,...,275.5,47.0,47.0,47,4.0,5.0,0,0.0,0.0,0
3,407-0001,1,9-Apr-18,Y,M,control,NaN,NaN,White,Not Hispanic or Latino,...,252.5,40.0,38.0,45.5,0.0,0.0,0,0.0,0.0,0
4,407-0002,2,4-Dec-17,Y,M,control,NaN,NaN,White,Not Hispanic or Latino,...,79.5,8.0,8.0,6.5,0.0,0.0,0,0.0,0.0,0


In [19]:
print("AG shape:", df_ag.shape)
print("RRV shape:", df_rrv.shape)

AG shape: (295, 31)
RRV shape: (295, 55)


In [21]:
print(df_ag.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295 entries, 0 to 294
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Participant                   283 non-null    object 
 1   ID                            295 non-null    int64  
 2   AG_Date                       278 non-null    object 
 3   Completed                     57 non-null     object 
 4   Gender                        280 non-null    object 
 5   Group                         260 non-null    object 
 6   Weight                        85 non-null     float64
 7   Height                        85 non-null     float64
 8   Race                          278 non-null    object 
 9   Ethnicity                     257 non-null    object 
 10  Age                           273 non-null    float64
 11  BMI                           85 non-null     float64
 12  Assmnt                        207 non-null    object 
 13  Avg_D

In [22]:
keep_cols = [
    "Participant","ID","Gender","Group","Race","Ethnicity","Age",
    "Assmnt",
    "Avg_Daily_Week_Min",
    "Avg_Daily_Sed_Week_Min",
    "Avg_Daily_Light_Week_Min",
    "Avg_Daily_Mod_Week_Min",
    "Avg_Daily_Vig_Week_Min",
    "Avg_Daily_Very_Vig_Week_Min"
]

df_ag = df_ag[keep_cols].copy()
print("Shape after column selection:", df_ag.shape)


Shape after column selection: (295, 14)


In [23]:
activity_cols = [
    "Avg_Daily_Week_Min",
    "Avg_Daily_Sed_Week_Min",
    "Avg_Daily_Light_Week_Min",
    "Avg_Daily_Mod_Week_Min",
    "Avg_Daily_Vig_Week_Min",
    "Avg_Daily_Very_Vig_Week_Min"
]

before = df_ag.shape[0]
df_ag = df_ag.dropna(subset=["Assmnt"] + activity_cols, how="all")
after = df_ag.shape[0]
print(f"Dropped {before - after} junk rows. New shape: {df_ag.shape}")

Dropped 88 junk rows. New shape: (207, 14)


In [24]:
df_ag["Assmnt"] = df_ag["Assmnt"].str.lower().str.strip()
mapping = {
    "baseline": "baseline",
    "1": "baseline",
    "endpsttr": "endposttr",
    "endposttr": "endposttr",
    "6": "endposttr",
    "postwash": "postwash",
    "10": "postwash",
}

df_ag["Assmnt"] = df_ag["Assmnt"].replace(mapping)
print("Assmnt value counts after cleaning:")
print(df_ag["Assmnt"].value_counts(dropna=False))

Assmnt value counts after cleaning:
Assmnt
baseline     84
pstwash      64
endposttr    59
Name: count, dtype: int64


In [25]:
import numpy as np

df_ag["Group"] = df_ag["Group"].str.lower().str.strip()

group_mapping = {
    "control": 0,
    "experimental": 1,
    "": np.nan,
    None: np.nan
}
df_ag["Group"] = df_ag["Group"].replace(group_mapping)
print("Group unique values:", df_ag["Group"].unique())

Group unique values: [ 0.  1. nan]


/var/folders/h5/x19gz3h54d1gw7mz5v_bflsh0000gn/T/ipykernel_8074/784456824.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_ag["Group"] = df_ag["Group"].replace(group_mapping)


In [26]:
for col in activity_cols:
    df_ag[col] = pd.to_numeric(df_ag[col], errors="coerce")

print(df_ag[activity_cols].dtypes)


Avg_Daily_Week_Min             float64
Avg_Daily_Sed_Week_Min         float64
Avg_Daily_Light_Week_Min       float64
Avg_Daily_Mod_Week_Min         float64
Avg_Daily_Vig_Week_Min         float64
Avg_Daily_Very_Vig_Week_Min    float64
dtype: object


In [27]:
before = df_ag.shape[0]
df_ag = df_ag.dropna(subset=activity_cols, how="any")
after = df_ag.shape[0]
print(f"Dropped {before - after} rows missing activity data. New shape: {df_ag.shape}")

Dropped 0 rows missing activity data. New shape: (207, 14)


In [28]:
df_ag["Gender"] = df_ag["Gender"].fillna(df_ag["Gender"].mode()[0])
df_ag["Race"] = df_ag["Race"].fillna(df_ag["Race"].mode()[0])
df_ag["Ethnicity"] = df_ag["Ethnicity"].fillna(df_ag["Ethnicity"].mode()[0])

In [29]:
df_ag["Age"] = df_ag["Age"].fillna(df_ag["Age"].median())
df_ag["Group"] = df_ag["Group"].fillna(df_ag["Group"].mode()[0])

In [30]:
print(df_ag.isna().sum())

Participant                    0
ID                             0
Gender                         0
Group                          0
Race                           0
Ethnicity                      0
Age                            0
Assmnt                         0
Avg_Daily_Week_Min             0
Avg_Daily_Sed_Week_Min         0
Avg_Daily_Light_Week_Min       0
Avg_Daily_Mod_Week_Min         0
Avg_Daily_Vig_Week_Min         0
Avg_Daily_Very_Vig_Week_Min    0
dtype: int64


In [31]:
print("Final shape:", df_ag.shape)
df_ag.head()

Final shape: (207, 14)


,Participant,ID,Gender,Group,Race,Ethnicity,Age,Assmnt,Avg_Daily_Week_Min,Avg_Daily_Sed_Week_Min,Avg_Daily_Light_Week_Min,Avg_Daily_Mod_Week_Min,Avg_Daily_Vig_Week_Min,Avg_Daily_Very_Vig_Week_Min
0,407-0001,1,M,0.0,White,Not Hispanic or Latino,32.0,baseline,910.0,629.0,256.0,22.0,3.0,0.0
2,407-0001,1,M,0.0,White,Not Hispanic or Latino,33.0,endposttr,824.0,521.0,252.0,47.0,4.0,0.0
3,407-0001,1,M,0.0,White,Not Hispanic or Latino,33.0,pstwash,848.0,579.0,229.0,40.0,0.0,0.0
4,407-0002,2,M,0.0,White,Not Hispanic or Latino,33.0,baseline,680.0,546.0,127.0,8.0,0.0,0.0
6,407-0002,2,M,0.0,White,Not Hispanic or Latino,33.0,endposttr,622.0,510.0,102.0,11.0,0.0,0.0


In [32]:
df_ag.to_csv("AGData_clean.csv", index=False)